# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** The decision is *"which pages should an editor refresh first?"* — an ordering under a tight time budget, not a yes/no per page. So the output is a per-page **refresh-priority score** that sorts the library into a ranked review queue. I *borrow* a binary decline label only to **evaluate** that ranking (precision@K), not because the task is classification: an editor never asks "is this page declining, yes/no?", they ask "what do I open first?"

In [1]:
import os, subprocess
import pandas as pd

# --- Bootstrap: locate the repo root so this runs from anywhere (local or Colab) ---
CSV = "data/raw/content_refresh_anonymized.csv"
REPO_URL = "https://github.com/thany-8/content-refresh-prioritizer"
REPO_DIR = "content-refresh-prioritizer"

def find_root(start, marker=CSV, up=6):
    """Walk up from `start` until a dir containing `marker` is found (repo root)."""
    d = os.path.abspath(start)
    for _ in range(up + 1):
        if os.path.exists(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    return None

root = find_root(os.getcwd())
if root is None:  # e.g. a fresh Colab VM: clone the public repo, then use it
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    root = os.path.abspath(REPO_DIR)
os.chdir(root)

assert os.path.exists(CSV), f"starter CSV not found under {root}"
df = pd.read_csv(CSV)

# The decision is "which pages first?" under a tight editor budget -> we need an ORDERING,
# so the output is a per-page priority SCORE that sorts pages into a ranked review queue.
n_pages = len(df)
n_clients = df["client_id"].nunique()
print(f"{n_pages:,} pages across {n_clients} clients.")
print("Editors review ~dozens of pages a week -> we must RANK, not label every page.")
print("=> Task type: ranking / scoring (priority score), judged by a top-of-queue metric.")

30,000 pages across 32 clients.
Editors review ~dozens of pages a week -> we must RANK, not label every page.
=> Task type: ranking / scoring (priority score), judged by a top-of-queue metric.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I score **refresh priority**. The label I can evaluate against on this slice is `is_declining_label = (trend_direction == "down")` — a **defined rule**, not an observed outcome. `trend_direction` is computed from `trend_pct` (last-30d vs prev-30d impressions), so the target is a **proxy** measured on *our own* traffic, **not** an observed Google action.

Two consequences I carry forward:
- **Leakage guard:** `trend_direction` and `trend_pct` can **never** be features — a model fed them just re-derives the label (that's what the W03 leakage notebook shows).
- **Careful claims:** because the label is a rule, I can only say the queue surfaces pages that *look like* our declining pages (**decision-support / directional**) — never that it predicts Google or *causes* recovery. The honest observed target (a forward-window decline) lives in the Week-3 warehouse; here I use the rule label as a stated proxy.

In [2]:
# The target is a RULE on our own data:  is_declining_label == (trend_direction == 'down').
proxy = df["trend_direction"].eq("down").astype(int)   # this IS is_declining_label
base_rate = proxy.mean() * 100
print(f"Label = is_declining_label = (trend_direction == 'down')  ->  base rate {base_rate:.1f}%")
print("Source: a defined rule on our 90-day traffic, not an observed Google action.")
print("So trend_direction & trend_pct are NEVER features, and claims stay decision-support.")

Label = is_declining_label = (trend_direction == 'down')  ->  base rate 54.2%
Source: a defined rule on our 90-day traffic, not an observed Google action.
So trend_direction & trend_pct are NEVER features, and claims stay decision-support.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**precision@K** — of the top-K pages the queue flags, what share are actually declining. I pick it because editor time is the scarce resource: only the **top of the list** has to be trustworthy, and *K* is literally how many pages an editor can review. Overall accuracy or ROC-AUC would reward being right about pages nobody will ever open.

**"Good" = beat the base rate at the top.** A no-skill queue scores ~**54.2%** (the share of pages that are 'down'). The cell below computes precision@K **today** on a simple, non-leaky baseline (staleness): it clears the base rate at K=1000 but only barely — which is exactly the gap a model has to widen in W07/W08.

In [3]:
# precision@K: of the top-K flagged pages, what share are truly 'down'.
base_rate = df["trend_direction"].eq("down").mean() * 100

def patk(frame, score_col, ascending, K):
    """precision@K for ranking `frame` by one column (drops rows missing that column)."""
    d = frame.dropna(subset=[score_col])
    top = d.sort_values(score_col, ascending=ascending).head(K)
    return top["trend_direction"].eq("down").mean() * 100

print(f"base rate (no-skill queue): {base_rate:.1f}%")
for K in (100, 500, 1000):
    # honest, non-label-derived signal: oldest-updated pages first
    p = patk(df, "days_since_last_update", False, K)
    print(f"  precision@{K} by staleness: {p:.1f}%")
print("K ties to editor capacity; only the top of the queue must be trustworthy.")

base rate (no-skill queue): 54.2%
  precision@100 by staleness: 37.0%
  precision@500 by staleness: 56.2%
  precision@1000 by staleness: 59.9%
K ties to editor capacity; only the top of the queue must be trustworthy.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (a page), inside one client.** 30,000 pages across 32 clients, `content_id` unique per row. The IDs are pseudonyms — I use `client_id` for grouped (client-holdout) splits, never as a feature.

In [4]:
# One row = one page (content item), nested under a client.
print("rows x cols:", df.shape)
print("content_id unique (one row per page):", df["content_id"].is_unique)
print("clients:", df["client_id"].nunique(),
      "| median pages per client:", int(df.groupby("client_id").size().median()))
print("IDs are pseudonyms -> grouping / client-holdout splits only, never features.\n")

# A few human-readable columns for one glance at the grain (no IDs / no client names):
readable = ["content_type", "word_count", "impressions_90d", "clicks_90d",
            "ctr", "avg_position", "days_since_last_update", "trend_direction"]
df[readable].head()

rows x cols: (30000, 44)
content_id unique (one row per page): True
clients: 32 | median pages per client: 567
IDs are pseudonyms -> grouping / client-holdout splits only, never features.



,content_type,word_count,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,trend_direction
0,keyword article,3221.0,3803,29,0.76,10.6,20,down
1,keyword article,2481.0,15320,7,0.05,20.3,25,down
2,keyword article,3515.0,12581,11,0.09,36.5,20,down
3,keyword article,NaN,11751,58,0.49,6.2,22,stable
4,keyword article,2803.0,19140,24,0.13,44.0,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Every **single** signal is a weak, disagreeing ranker: staleness clears the base rate by a hair, low CTR lands *below* it, worst-position (once the `avg_position == 0` "no-data" sentinel is excluded) is worse still. The decline signal is real but **spread thin across many tangled, partly-missing features** — systematic missingness along `content_type`, ×100 rate columns, sentinel values — so no hand-written if-statement ranks well. That's the case for a model that **combines** the weak signals. But ML only earns its place if it **beats this rule baseline**, which is exactly what W07 (baseline) and W08 (model) put to the test.

In [5]:
# Each lone rule barely moves off the base rate, and they disagree -> combine signals with a model.
real_pos = df[df["avg_position"] > 0]   # avg_position == 0 means "no data", not a real rank
rows = [
    ("base rate (no skill)",         base_rate),
    ("rank by staleness",            patk(df, "days_since_last_update", False, 1000)),
    ("rank by low CTR",              patk(df, "ctr", True, 1000)),
    ("rank by worst real position",  patk(real_pos, "avg_position", False, 1000)),
]
summary = pd.DataFrame(rows, columns=["single rule", "precision@1000 (%)"]).round(1)
print(summary.to_string(index=False))
print("\nNo single rule dominates -> a model must combine signals AND beat this baseline (W07/W08).")

                single rule  precision@1000 (%)
       base rate (no skill)                54.2
          rank by staleness                59.9
            rank by low CTR                50.2
rank by worst real position                33.8

No single rule dominates -> a model must combine signals AND beat this baseline (W07/W08).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.